<a href="https://colab.research.google.com/github/Octaxx/DLI-Assignment/blob/main/Kaifung_Model_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import requests
import nbformat
from IPython import get_ipython

# Load and run DatasetCleaning.ipynb from GitHub (including Step 7)
url = "https://raw.githubusercontent.com/Octaxx/DLI-Assignment/refs/heads/main/DatasetCleaning.ipynb"
response = requests.get(url)
notebook = nbformat.reads(response.text, as_version=4)
ipython = get_ipython()

print("⚙️ Running cells from DatasetCleaning.ipynb...\n")

for i, cell in enumerate(notebook.cells):
    if cell.cell_type == 'code':
        try:
            print(f"▶️ Executing cell {i+1}...")
            ipython.run_cell(cell.source)
        except Exception as e:
            print(f"❌ Error in cell {i+1}: {e}")

print("\n✅ All notebook cells executed.")

⚙️ Running cells from DatasetCleaning.ipynb...

▶️ Executing cell 2...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
▶️ Executing cell 3...
📊 BEFORE BALANCING
--------------------------------------------------
Total rows before balancing: 18634
Class balance before balancing:
Email Type
Safe Email        11322
Phishing Email     7312
Name: count, dtype: int64


,Email Text,Email Type,Label
0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email,0
1,the other side of * galicismos * * galicismo *...,Safe Email,0
2,re : equistar deal tickets are you still avail...,Safe Email,0
3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email,1
4,software at incredibly low prices ( 86 % lower...,Phishing Email,1
5,global risk management operations sally congra...,Safe Email,0
6,"On Sun, Aug 11, 2002 at 11:17:47AM +0100, wint...",Safe Email,0
7,"entourage , stockmogul newsletter ralph velez ...",Phishing Email,1
8,"we owe you lots of money dear applicant , afte...",Phishing Email,1
9,re : coastal deal - with exxon participation u...,Safe Email,0


▶️ Executing cell 4...

📊 AFTER BALANCING (Oversampling)
--------------------------------------------------
Total rows after balancing: 22644
Class balance after balancing:
Email Type
Phishing Email    11322
Safe Email        11322
Name: count, dtype: int64
▶️ Executing cell 5...

🧾 SAMPLE OF CLEANED & BALANCED DATAFRAME
Total Rows        : 22644
Phishing Emails   : 11322
Safe Emails       : 11322

🧪 Cleaned & Balanced DataFrame (First 5 Rows):


,Email Text,Email Type,Label
0,INVESTMENT SCHOLARS CLUB- bringing you the lat...,Phishing Email,1
1,semantics : il dominio tempo-aspettuale il dom...,Safe Email,0
2,mature mom and her young horny lover ! . . woo...,Phishing Email,1
3,do you own a car ; starting december 7 th ford...,Phishing Email,1
4,rescue you from highprice medicaments and badp...,Phishing Email,1



🎯 Phishing Emails (First 5):


...


✅ Safe Emails (First 5):


,Email Text,Email Type,Label
1,semantics : il dominio tempo-aspettuale il dom...,Safe Email,0
5,http://www.bbc.co.uk/radio1/alt/nireland/ni_te...,Safe Email,0
6,Hi Damian.SuSe has a Sparc version I previousl...,Safe Email,0
7,"hpl nom for may 25 , 2001 ( see attached file ...",Safe Email,0
8,iatl 14 : final cfp the 14th annual meeting - ...,Safe Email,0


▶️ Executing cell 6...

🧠 Sample Extracted Features (First 5):


,Email Preview,char_count,word_count,exclamation_count,uppercase_ratio,has_link,has_login_word,has_html
0,INVESTMENT SCHOLARS CLUB- bringing you the lat...,3893,586,2,0.034,0,1,0
1,semantics : il dominio tempo-aspettuale il dom...,904,154,0,0.000,0,0,0
2,mature mom and her young horny lover ! . . woo...,664,141,1,0.000,0,0,0
3,do you own a car ; starting december 7 th ford...,792,170,0,0.000,0,0,0
4,rescue you from highprice medicaments and badp...,796,163,0,0.000,0,0,0



✅ Final Columns:
['Email Text', 'Email Type', 'Label', 'Email Preview', 'char_count', 'word_count', 'exclamation_count', 'uppercase_ratio', 'has_link', 'has_login_word', 'has_html']

✅ All notebook cells executed.


In [ ]:
# =========================
# 0) Imports & Config
# =========================
import tempfile
import os, io, re, time, joblib, numpy as np, pandas as pd
from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import VotingClassifier

import tensorflow as tf
from tensorflow.keras import Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# =========================================
# Helpers to store a single Keras model in PKL
# =========================================
def serialize_keras_h5(model) -> bytes:
    """Serialize a Keras model to bytes using a temp .h5 file (Keras 3 compatible)."""
    with tempfile.TemporaryDirectory() as tmpd:
        path = os.path.join(tmpd, "final_mlp.h5")   # or "final_mlp.keras"
        model.save(path)                             # Keras 3: no save_format arg
        with open(path, "rb") as f:
            return f.read()

def deserialize_keras_h5(model_bytes):
    """Load a Keras model from bytes by writing to a temp .h5 file (Keras 3 compatible)."""
    with tempfile.TemporaryDirectory() as tmpd:
        path = os.path.join(tmpd, "final_mlp.h5")    # use same extension you serialized with
        with open(path, "wb") as f:
            f.write(model_bytes)
        return tf.keras.models.load_model(path)

# =========================
# 1) Data & basic cleaning
# =========================
# Expect df_balanced defined in your notebook with columns: 'Email Text' and 'Label'
kf_df_balanced = df_balanced.copy()


# =========================
# 2) Custom features
# =========================
def has_login_word_fn(s):
    words = ['login','signin','verify','reset','account','password','bank','unlock','security','update']
    return int(any(w in s for w in words))

def build_custom_features(df, text_col='Email Text'):
    txt = df[text_col].fillna("")
    if 'char_count' not in df: df['char_count'] = txt.str.len()
    if 'word_count' not in df: df['word_count'] = txt.str.split().str.len()
    if 'exclamation_count' not in df: df['exclamation_count'] = txt.str.count('!')
    if 'uppercase_ratio' not in df:
        raw = df[text_col].fillna("")
        up = raw.apply(lambda s: sum(1 for c in s if c.isupper()))
        ln = raw.str.len().replace(0, 1)
        df['uppercase_ratio'] = up / ln
    if 'has_link' not in df: df['has_link'] = txt.str.contains(r'<url>').astype(int)
    if 'has_login_word' not in df: df['has_login_word'] = txt.apply(has_login_word_fn).astype(int)
    if 'has_html' not in df: df['has_html'] = txt.str.contains(r'<html|</html|<a |<div|<span', regex=True).astype(int)
    # extras
    if 'digit_ratio' not in df:
        raw = df[text_col].fillna("")
        digits = raw.apply(lambda s: sum(ch.isdigit() for ch in s))
        ln = raw.str.len().replace(0, 1)
        df['digit_ratio'] = digits / ln
    if 'avg_word_len' not in df:
        df['avg_word_len'] = txt.apply(lambda s: (sum(len(w) for w in s.split()) / (len(s.split()) or 1)))
    return df

kf_df_balanced = build_custom_features(kf_df_balanced)

base_custom_features = ['char_count','word_count','exclamation_count',
                        'uppercase_ratio','has_link','has_login_word','has_html']
extra_feats = ['digit_ratio','avg_word_len']
custom_features = [f for f in base_custom_features + extra_feats if f in kf_df_balanced.columns]
print("Using custom features:", custom_features)

# =========================
# 3) MLP model (for meta feature)
# =========================
def create_mlp_model(input_dim, lr=1e-3):
    kf_model = Sequential([
        Input(shape=(input_dim,)),
        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    kf_model.compile(optimizer=Adam(learning_rate=lr),
                     loss='binary_crossentropy',
                     metrics=['accuracy'])
    return kf_model

# =========================
# 4) OOF MLP probabilities
# =========================
def oof_mlp_probs(X_dense_train, kf_y_train, X_dense_test, n_splits=5, epochs=20, batch=32):
    """Create OOF train probs and averaged test probs (no model saving here)."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof_train = np.zeros((X_dense_train.shape[0], 1), dtype=np.float32)
    test_fold_preds = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_dense_train, kf_y_train), 1):
        print(f"\n🧪 Training MLP (Fold {fold}/{n_splits})")
        Xtr, Xva = X_dense_train[tr_idx], X_dense_train[va_idx]
        ytr, yva = kf_y_train[tr_idx], kf_y_train[va_idx]

        kf_model = create_mlp_model(input_dim=Xtr.shape[1])
        es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1)
        rlrop = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)

        kf_model.fit(Xtr, ytr,
                     validation_data=(Xva, yva),
                     epochs=epochs, batch_size=batch,
                     verbose=1, callbacks=[es, rlrop])

        oof_train[va_idx] = kf_model.predict(Xva, verbose=0)
        test_fold_preds.append(kf_model.predict(X_dense_test, verbose=0))

    test_mean = np.mean(test_fold_preds, axis=0)
    return oof_train, test_mean

# =========================
# 5) Train/Eval loop + pick best run
# =========================
num_runs = 2
n_epochs = 10

kf_results = []
per_run_artifacts = []  # store objects from each run so we can save the best

for run in range(num_runs):
    print(f"\n📦 Hybrid Training Run {run+1}/{num_runs}")
    kf_X = kf_df_balanced[['Email Text'] + custom_features].copy()
    kf_y = kf_df_balanced['Label'].values.astype(int)

    kf_X_train, kf_X_test, kf_y_train, kf_y_test = train_test_split(
        kf_X, kf_y, test_size=0.2, stratify=kf_y, random_state=SEED + run)

    # TF-IDF for NB/SVM branch
    tfidf = TfidfVectorizer(stop_words='english',
                            ngram_range=(1,3),
                            max_features=15000,
                            min_df=2, max_df=0.9,
                            sublinear_tf=True)
    Xtr_text = tfidf.fit_transform(kf_X_train['Email Text'])
    Xte_text = tfidf.transform(kf_X_test['Email Text'])

    # Scale numeric features (non-negative for NB)
    feat_scaler = MinMaxScaler()
    Xtr_feats = feat_scaler.fit_transform(kf_X_train[custom_features].values)
    Xte_feats = feat_scaler.transform(kf_X_test[custom_features].values)

    # MLP input: dense = [TF-IDF | scaled custom]
    Xtr_dense_for_mlp = np.hstack([Xtr_text.toarray(), Xtr_feats])
    Xte_dense_for_mlp = np.hstack([Xte_text.toarray(), Xte_feats])

    # OOF MLP probabilities for stacking
    oof_train_probs, test_probs = oof_mlp_probs(
        Xtr_dense_for_mlp, kf_y_train, Xte_dense_for_mlp,
        n_splits=5, epochs=n_epochs, batch=32
    )

    # ALSO: train ONE final MLP on the full training dense set for deployment
    print("\n🧱 Training FINAL MLP on FULL training set")
    final_mlp = create_mlp_model(input_dim=Xtr_dense_for_mlp.shape[1])
    es_final = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1)
    rlr_final = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)

    final_mlp.fit(
        Xtr_dense_for_mlp, kf_y_train,
        validation_split=0.1,
        epochs=n_epochs, batch_size=32,
        verbose=1, callbacks=[es_final, rlr_final]
    )
    final_mlp_bytes = serialize_keras_h5(final_mlp)

    # Stack for NB/SVM: [TFIDF | custom feats | MLP prob]
    Xtr_stack = hstack([Xtr_text, csr_matrix(Xtr_feats), csr_matrix(oof_train_probs)])
    Xte_stack = hstack([Xte_text, csr_matrix(Xte_feats), csr_matrix(test_probs)])

    # Train NB (quick alpha search)
    nb_gs = GridSearchCV(MultinomialNB(), {"alpha":[0.1,0.3,0.5,1.0]}, cv=3, n_jobs=-1)
    nb_gs.fit(Xtr_stack, kf_y_train)
    nb = nb_gs.best_estimator_

    # Train LinearSVC + calibration
    svm_base = LinearSVC(C=1.0, random_state=SEED)
    svm = CalibratedClassifierCV(svm_base, cv=3)
    svm.fit(Xtr_stack, kf_y_train)

    # Soft voting
    ensemble = VotingClassifier(estimators=[('nb', nb), ('svm', svm)], voting='soft')
    ensemble.fit(Xtr_stack, kf_y_train)

    # Evaluate default 0.5 threshold
    kf_y_proba = ensemble.predict_proba(Xte_stack)[:, 1]
    kf_y_pred  = (kf_y_proba >= 0.5).astype(int)

    kf_acc  = accuracy_score(kf_y_test, kf_y_pred)
    kf_prec = precision_score(kf_y_test, kf_y_pred)
    kf_rec  = recall_score(kf_y_test, kf_y_pred)
    kf_f1   = f1_score(kf_y_test, kf_y_pred)
    kf_roc  = roc_auc_score(kf_y_test, kf_y_proba)
    print(f"Default t=0.50 -> Acc {kf_acc:.4f}  Prec {kf_prec:.4f}  Rec {kf_rec:.4f}  F1 {kf_f1:.4f}  ROC-AUC {kf_roc:.4f}")

    # Threshold tuning (optimize accuracy; change to F1 if preferred)
    best_t, best_metric = 0.5, kf_acc
    for t in np.linspace(0.30, 0.70, 41):
        yp = (kf_y_proba >= t).astype(int)
        metric = accuracy_score(kf_y_test, yp)
        if metric > best_metric:
            best_metric, best_t = metric, t

    if best_t != 0.5:
        yp = (kf_y_proba >= best_t).astype(int)
        kf_acc  = accuracy_score(kf_y_test, yp)
        kf_prec = precision_score(kf_y_test, yp)
        kf_rec  = recall_score(kf_y_test, yp)
        kf_f1   = f1_score(kf_y_test, yp)
        print(f"Best threshold t={best_t:.2f} -> Acc {kf_acc:.4f}  Prec {kf_prec:.4f}  Rec {kf_rec:.4f}  F1 {kf_f1:.4f}")

    # store run results + artifacts
    kf_results.append({'run': run+1, 'acc': kf_acc, 'prec': kf_prec, 'rec': kf_rec, 'f1': kf_f1, 'roc_auc': kf_roc})
    per_run_artifacts.append({
        "tfidf": tfidf,
        "feat_scaler": feat_scaler,
        "custom_features": custom_features,
        "nb": nb,
        "svm": svm,
        "ensemble": ensemble,
        "best_threshold": float(best_t),
        "final_mlp_bytes": final_mlp_bytes
    })





Using custom features: ['char_count', 'word_count', 'exclamation_count', 'uppercase_ratio', 'has_link', 'has_login_word', 'has_html', 'digit_ratio', 'avg_word_len']

📦 Hybrid Training Run 1/2

🧪 Training MLP (Fold 1/5)
Epoch 1/10
453/453 ━━━━━━━━━━━━━━━━━━━━ 18s 31ms/step - accuracy: 0.8937 - loss: 0.2334 - val_accuracy: 0.9514 - val_loss: 0.1484 - learning_rate: 0.0010
Epoch 2/10
453/453 ━━━━━━━━━━━━━━━━━━━━ 20s 31ms/step - accuracy: 0.9808 - loss: 0.0526 - val_accuracy: 0.9757 - val_loss: 0.0573 - learning_rate: 0.0010
Epoch 3/10
453/453 ━━━━━━━━━━━━━━━━━━━━ 20s 30ms/step - accuracy: 0.9885 - loss: 0.0279 - val_accuracy: 0.9752 - val_loss: 0.0745 - learning_rate: 0.0010
Epoch 4/10
452/453 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.9899 - loss: 0.0231
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
453/453 ━━━━━━━━━━━━━━━━━━━━ 21s 31ms/step - accuracy: 0.9899 - loss: 0.0231 - val_accuracy: 0.9713 - val_loss: 0.0849 - learning_rate: 0.0010
Epoch 5/10
45

Default t=0.50 -> Acc 0.9834  Prec 0.9744  Rec 0.9929  F1 0.9836  ROC-AUC 0.9987
Best threshold t=0.53 -> Acc 0.9839  Prec 0.9753  Rec 0.9929  F1 0.9840

📦 Hybrid Training Run 2/2

🧪 Training MLP (Fold 1/5)
Epoch 1/10
453/453 ━━━━━━━━━━━━━━━━━━━━ 18s 33ms/step - accuracy: 0.9041 - loss: 0.2223 - val_accuracy: 0.9765 - val_loss: 0.1052 - learning_rate: 0.0010
Epoch 2/10
453/453 ━━━━━━━━━━━━━━━━━━━━ 14s 31ms/step - accuracy: 0.9811 - loss: 0.0500 - val_accuracy: 0.9746 - val_loss: 0.0781 - learning_rate: 0.0010
Epoch 3/10
453/453 ━━━━━━━━━━━━━━━━━━━━ 14s 31ms/step - accuracy: 0.9851 - loss: 0.0315 - val_accuracy: 0.9752 - val_loss: 0.0742 - learning_rate: 0.0010
Epoch 4/10
453/453 ━━━━━━━━━━━━━━━━━━━━ 21s 32ms/step - accuracy: 0.9863 - loss: 0.0284 - val_accuracy: 0.9763 - val_loss: 0.0792 - learning_rate: 0.0010
Epoch 5/10
452/453 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.9886 - loss: 0.0267
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
453/453 ━━━━━━

Default t=0.50 -> Acc 0.9821  Prec 0.9687  Rec 0.9965  F1 0.9824  ROC-AUC 0.9986
Best threshold t=0.70 -> Acc 0.9830  Prec 0.9732  Rec 0.9934  F1 0.9832


In [ ]:
# =========================
# 6) Pick best run & SAVE single .pkl
# =========================
res_df = pd.DataFrame(kf_results)
print("\n=== Summary over runs ===")
print(res_df.describe()[['acc','prec','rec','f1','roc_auc']].loc[['mean','std','min','max']])

best_idx = int(res_df['acc'].idxmax())
best_art = per_run_artifacts[best_idx]
print(f"\n✅ Saving best run (run #{best_idx+1}) with Acc={res_df.loc[best_idx,'acc']:.4f}")

# 👉 make this a full filename, not a folder
bundle_path = '/content/drive/My Drive/Colab Notebooks/PhishingModel/Kaifung/KaifungModel.pkl'

kf_bundle = {
    "results": kf_results,                 # metrics from ALL runs
    # preprocessing & features from the BEST run
    "tfidf": best_art["tfidf"],
    "feat_scaler": best_art["feat_scaler"],
    "custom_features": best_art["custom_features"],
    # models from the BEST run
    "nb": best_art["nb"],
    "svm": best_art["svm"],
    "ensemble": best_art["ensemble"],      # VotingClassifier (NB+SVM)
    "final_mlp_bytes": best_art["final_mlp_bytes"],  # single final MLP (bytes)
    # decision rule from the BEST run
    "best_threshold": best_art["best_threshold"],
}

joblib.dump(kf_bundle, bundle_path, compress=3)
print("💾 Saved ALL-IN-ONE bundle to:", bundle_path)


=== Summary over runs ===
           acc      prec       rec        f1   roc_auc
mean  0.983440  0.974221  0.993154  0.983596  0.998644
std   0.000625  0.001484  0.000312  0.000603  0.000009
min   0.982998  0.973172  0.992933  0.983169  0.998637
max   0.983882  0.975271  0.993375  0.984023  0.998650

✅ Saving best run (run #1) with Acc=0.9839
💾 Saved ALL-IN-ONE bundle to: /content/drive/My Drive/Colab Notebooks/PhishingModel/Kaifung/KaifungModel.pkl
